# Ingest: mathayom-math-corpus -> Unified Schema Parquet

**Source:** https://github.com/mingrath/mathayom-math-corpus  
**Language:** Thai only (filtered by manifest.tsv `lang == Thai`)  
**Output schema:** `id, source, license, text, n_chars, n_words, quality_flags, dup_group, domain`  
**Domain:** `math`

### What the repo contains
- 30 `.txt` files (OCR from PDFs) in `txt/`
- `manifest.tsv` with columns: `id, lang, level, license, url`
- 16 files tagged `lang=Thai` (M1-M3 math textbooks, lesson plans, workbooks)
- 14 files tagged `lang=English` - **skipped**

### Estimated runtime
| Step | Time |
|------|------|
| git clone | 5-30 sec |
| parse + clean all 16 Thai files | 2-5 sec |
| write parquet | < 1 sec |
| **Total** | **< 1 minute** |

In [ ]:
# Cell 1 - config
REPO_URL   = "https://github.com/mingrath/mathayom-math-corpus.git"
CLONE_DIR  = "/tmp/mathayom-math-corpus"   # local clone target
OUTPUT_DIR = "/mnt/ssd/shards/mathayom"    # output parquet directory
SOURCE_TAG = "mathayom"                    # value for 'source' field
SHARD_SIZE = 5_000                         # docs per output parquet shard

MIN_PARA_CHARS  = 30    # drop paragraphs shorter than this
THAI_RATIO_WARN = 0.15  # paragraphs below this get 'low_thai' flag

In [ ]:
# Cell 2 - install deps (skip if already installed)
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyarrow>=16.0", "tqdm"],
    check=True,
)

In [ ]:
# Cell 3 - clone repo
import subprocess
from pathlib import Path

clone_path = Path(CLONE_DIR)

if clone_path.exists():
    print("Already cloned. Pulling latest...")
    result = subprocess.run(
        ["git", "-C", str(clone_path), "pull", "--rebase"],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or "Already up to date.")
else:
    print(f"Cloning {REPO_URL}")
    subprocess.run(
        ["git", "clone", "--depth=1", REPO_URL, str(clone_path)],
        check=True,
    )

txt_dir = clone_path / "txt"
print(f"txt/ files: {len(list(txt_dir.glob('*.txt')))}")
print("Done.")

In [ ]:
# Cell 4 - read manifest, keep Thai files only
import csv
from pathlib import Path

manifest_path = Path(CLONE_DIR) / "manifest.tsv"
txt_dir       = Path(CLONE_DIR) / "txt"

thai_files    = []
english_count = 0

with open(manifest_path, encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        file_path = txt_dir / f"{row['id']}.txt"
        if row["lang"].strip().lower() == "thai":
            if file_path.exists():
                thai_files.append({
                    "id":      row["id"],
                    "level":   row["level"],
                    "license": row["license"],
                    "url":     row["url"],
                    "path":    file_path,
                })
            else:
                print(f"WARNING: file not found for {row['id']}")
        else:
            english_count += 1

print(f"Thai files   : {len(thai_files)}")
print(f"English files: {english_count} (skipped)")
print()
print(f"{'ID':<40} {'Level':<12} {'License'}")
print("-" * 70)
for fi in thai_files:
    print(f"{fi['id']:<40} {fi['level']:<12} {fi['license']}")

In [ ]:
# Cell 5 - text cleaning helpers
import re
import unicodedata

_THAI_RE = re.compile(r"[\u0e00-\u0e7f]")
_ZW      = "\u200b\u200c\u200d\ufeff"

# Math signal: LaTeX + Thai math vocabulary
_MATH_RE = re.compile(
    r"(\$|\\frac|\\sum|\\int|\\begin\{"
    r"|[0-9]+\s*[\+\-\*/=]\s*[0-9]+"
    r"|\u0e42\u0e08\u0e17\u0e22\u0e4c"   # โจทย์
    r"|\u0e04\u0e33\u0e15\u0e2d\u0e1a"   # คำตอบ
    r"|\u0e2a\u0e21\u0e01\u0e32\u0e23"   # สมการ
    r"|\u0e1f\u0e31\u0e07\u0e01\u0e4c\u0e0a\u0e31\u0e19"  # ฟังก์ชัน
    r"|\u0e08\u0e33\u0e19\u0e27\u0e19"   # จำนวน
    r"|\u0e40\u0e28\u0e29\u0e2a\u0e48\u0e27\u0e19"  # เศษส่วน
    r"|\u0e23\u0e49\u0e2d\u0e22\u0e25\u0e30"  # ร้อยละ
    r"|\u0e01\u0e23\u0e32\u0e1f"         # กราฟ
    r"|\u0e21\u0e38\u0e21"               # มุม
    r"|\u0e1e\u0e37\u0e49\u0e19\u0e17\u0e35\u0e48"  # พื้นที่
    r")",
    re.IGNORECASE,
)

def clean_thai_text(text):
    """NFC normalize, strip zero-width chars, remove OCR replacement chars."""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(str.maketrans("", "", _ZW))
    text = text.replace("\ufffd", "")        # OCR replacement char from bad PDF scan
    text = re.sub(r"\n{3,}", "\n\n", text)  # collapse excess blank lines
    return text.strip()

def thai_ratio(text):
    if not text:
        return 0.0
    return len(_THAI_RE.findall(text)) / len(text)

def has_math(text):
    return bool(_MATH_RE.search(text))

def split_paragraphs(text, min_chars=30):
    """Split on blank lines; drop very short segments."""
    paras = re.split(r"\n{2,}", text)
    return [p.strip() for p in paras if len(p.strip()) >= min_chars]

# Sanity check
_t = "\u0e08\u0e33\ufffd\u0e19\u0e27\u0e19\u200b\u0e40\u0e15\u0e47\u0e21"  # จำ<bad>นวน<zwsp>เต็ม
print("OCR clean test:", repr(clean_thai_text(_t)))
print("Thai ratio test:", round(thai_ratio("คณิตศาสตร์ math 123"), 3))
print("Helpers loaded.")

In [ ]:
# Cell 6 - peek at one Thai file to verify cleaning
sample_meta = thai_files[2]
raw         = sample_meta["path"].read_text(encoding="utf-8", errors="replace")
cleaned     = clean_thai_text(raw)
paras       = split_paragraphs(cleaned, MIN_PARA_CHARS)

print(f"File     : {sample_meta['id']}")
print(f"Level    : {sample_meta['level']}")
print(f"License  : {sample_meta['license']}")
print(f"Raw chars: {len(raw):,}  ->  cleaned: {len(cleaned):,}")
print(f"Paragraphs after split: {len(paras)}")
print(f"Thai ratio (full doc): {thai_ratio(cleaned):.3f}")
print()
print("--- First 3 paragraphs ---")
for i, p in enumerate(paras[:3]):
    print(f"[{i}] ({len(p)} chars, thai={thai_ratio(p):.2f})")
    print(p[:300])
    print()

In [ ]:
# Cell 7 - convert all Thai files to unified schema docs
from tqdm.auto import tqdm

all_docs       = []
stats_per_file = []
doc_index      = 0

for file_meta in tqdm(thai_files, desc="processing Thai files"):
    raw     = file_meta["path"].read_text(encoding="utf-8", errors="replace")
    cleaned = clean_thai_text(raw)
    paras   = split_paragraphs(cleaned, MIN_PARA_CHARS)

    file_docs = []
    for para in paras:
        tr             = thai_ratio(para)
        quality_flags  = []
        if tr < THAI_RATIO_WARN:
            quality_flags.append("low_thai")
        if not has_math(para):
            quality_flags.append("no_math_signal")

        doc = {
            "id":            f"{SOURCE_TAG}_{file_meta['id']}_{doc_index:06d}",
            "source":        SOURCE_TAG,
            "license":       file_meta["license"],
            "text":          para,
            "n_chars":       len(para),
            "n_words":       len(para.split()),
            "quality_flags": quality_flags,
            "dup_group":     None,
            "domain":        "math",
            "meta": {
                "source_file": file_meta["id"],
                "level":       file_meta["level"],
                "url":         file_meta["url"],
                "thai_ratio":  round(tr, 4),
            },
        }
        file_docs.append(doc)
        doc_index += 1

    stats_per_file.append({
        "file":     file_meta["id"],
        "level":    file_meta["level"],
        "license":  file_meta["license"],
        "paras":    len(file_docs),
        "low_thai": sum(1 for d in file_docs if "low_thai"       in d["quality_flags"]),
        "no_math":  sum(1 for d in file_docs if "no_math_signal" in d["quality_flags"]),
    })
    all_docs.extend(file_docs)

print(f"\nTotal docs : {len(all_docs):,}")
print(f"Avg chars  : {sum(d['n_chars'] for d in all_docs) // max(len(all_docs), 1):,}")
print()
print(f"{'File':<45} {'Level':<12} {'Paras':>6} {'LowThai':>8} {'NoMath':>8}")
print("-" * 85)
for s in stats_per_file:
    print(f"{s['file']:<45} {s['level']:<12} {s['paras']:>6,} {s['low_thai']:>8,} {s['no_math']:>8,}")

In [ ]:
# Cell 8 - spot-check 5 random docs
import random

for i, doc in enumerate(random.sample(all_docs, min(5, len(all_docs)))):
    print(f"--- Doc {i+1} ---")
    print(f"id            : {doc['id']}")
    print(f"n_chars       : {doc['n_chars']}")
    print(f"license       : {doc['license']}")
    print(f"quality_flags : {doc['quality_flags']}")
    print(f"thai_ratio    : {doc['meta']['thai_ratio']}")
    print(f"text preview  :")
    print(doc["text"][:300])
    print()

In [ ]:
# Cell 9 - write to parquet shards
import math
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

out_path = Path(OUTPUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

n_shards = max(1, math.ceil(len(all_docs) / SHARD_SIZE))
written  = 0

for i in range(n_shards):
    chunk    = all_docs[i * SHARD_SIZE : (i + 1) * SHARD_SIZE]
    shard_fn = out_path / f"mathayom_{i:04d}.parquet"
    table    = pa.Table.from_pylist(chunk)
    pq.write_table(table, shard_fn, compression="snappy")
    written += len(chunk)
    print(f"  Shard {i:04d}: {len(chunk):,} docs -> {shard_fn.name}")

print(f"\nDone. {written:,} docs in {n_shards} shard(s) -> {out_path}")

In [ ]:
# Cell 10 - verify: read back first shard
import json
import pyarrow.parquet as pq
from pathlib import Path

shards = sorted(Path(OUTPUT_DIR).glob("*.parquet"))
if not shards:
    print("ERROR: no parquet files in output dir")
else:
    table = pq.read_table(shards[0])
    print(f"Schema of {shards[0].name}:")
    print(table.schema)
    print(f"\nRows in first shard: {len(table):,}")
    print("\nFirst doc:")
    print(json.dumps(table.to_pylist()[0], ensure_ascii=False, indent=2))

In [ ]:
# Cell 11 - summary log for Google Sheet / handoff doc
import json
import pyarrow.parquet as pq
from datetime import datetime
from pathlib import Path

shards      = sorted(Path(OUTPUT_DIR).glob("*.parquet"))
total_rows  = sum(pq.read_metadata(s).num_rows for s in shards)
total_bytes = sum(s.stat().st_size for s in shards)

log = {
    "source":             SOURCE_TAG,
    "repo":               REPO_URL,
    "processed_at":       datetime.now().isoformat(),
    "lang_filter":        "Thai only (manifest.tsv lang==Thai)",
    "thai_files_used":    len(thai_files),
    "english_skipped":    english_count,
    "n_shards":           len(shards),
    "n_docs":             total_rows,
    "size_mb":            round(total_bytes / 1e6, 2),
    "output_dir":         str(Path(OUTPUT_DIR)),
    "schema":             "id,source,license,text,n_chars,n_words,quality_flags,dup_group,domain",
    "domain":             "math",
    "quality_flags_used": ["low_thai (thai_ratio<0.15)", "no_math_signal"],
    "cleaning":           "NFC + zero-width strip + U+FFFD OCR artifact removal",
    "granularity":        "paragraph-level (split on double newline)",
}

print(json.dumps(log, ensure_ascii=False, indent=2))

---

## Part 2: thai-math-exams-dataset

**Source:** https://github.com/mingrath/thai-math-exams-dataset  
**Content:** 769+ Q&A records from O-NET, PISA, Olympiad, blogs, YouTube, portals  
**Format:** JSONL files in `dataset/` - fields: `question, choices, answer, solution, grade, topic, exam_type, year, license_note`  
**Strategy:** join question + choices + answer + solution into one training text per record

### Estimated runtime
| Step | Time |
|------|------|
| git clone | 5-15 sec |
| parse all JSONL files | < 2 sec |
| write parquet | < 1 sec |
| **Total** | **< 30 seconds** |

In [ ]:
# Cell 12 - config for exams dataset
EXAMS_REPO_URL  = "https://github.com/mingrath/thai-math-exams-dataset.git"
EXAMS_CLONE_DIR = "/tmp/thai-math-exams-dataset"
EXAMS_OUT_DIR   = "/mnt/ssd/shards/thai_math_exams"
EXAMS_SOURCE    = "thai_math_exams"

# JSONL files to load from dataset/ (all of them)
EXAMS_JSONL_FILES = [
    "onet-niets.jsonl",
    "onet-niets_partial.jsonl",
    "official-extra.jsonl",
    "portals.jsonl",
    "blogs.jsonl",
    "blogs_partial.jsonl",
    "youtube.jsonl",
    "youtube_partial.jsonl",
    "wordwall.jsonl",
    "pdf-repos.jsonl",
    "pdf-repos_partial.jsonl",
]

In [ ]:
# Cell 13 - clone exams dataset repo
import subprocess
from pathlib import Path

exams_path = Path(EXAMS_CLONE_DIR)

if exams_path.exists():
    print("Already cloned. Pulling latest...")
    result = subprocess.run(
        ["git", "-C", str(exams_path), "pull", "--rebase"],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or "Already up to date.")
else:
    print(f"Cloning {EXAMS_REPO_URL}")
    subprocess.run(
        ["git", "clone", "--depth=1", EXAMS_REPO_URL, str(exams_path)],
        check=True,
    )

dataset_dir = exams_path / "dataset"
found = list(dataset_dir.glob("*.jsonl"))
print(f"JSONL files in dataset/: {len(found)}")
for f in sorted(found):
    print(f"  {f.name:<35} {f.stat().st_size // 1024} KB")
print("Done.")

In [ ]:
# Cell 14 - peek at one record to verify schema
import json
from pathlib import Path

sample_file = Path(EXAMS_CLONE_DIR) / "dataset" / "onet-niets.jsonl"
with open(sample_file, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            obj = json.loads(line)
            print("Keys:", list(obj.keys()))
            print()
            print(json.dumps(obj, ensure_ascii=False, indent=2))
            break

In [ ]:
# Cell 15 - parse all JSONL files -> unified schema docs
# Each record: question + choices + answer + solution joined as one training text
import json
import re
import unicodedata
from pathlib import Path
from tqdm.auto import tqdm

_THAI_RE = re.compile(r"[฀-๿]")
_ZW      = "​‌‍﻿"

def _clean(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(str.maketrans("", "", _ZW))
    text = text.replace("�", "")
    return text.strip()

def _thai_ratio(text):
    if not text:
        return 0.0
    return len(_THAI_RE.findall(text)) / len(text)

def _build_text(obj):
    """Join question + choices + answer + solution into one readable training doc."""
    parts = []

    q = _clean(obj.get("question", ""))
    if q:
        parts.append(f"โจทย์: {q}")

    choices = obj.get("choices") or []
    if choices and isinstance(choices, list):
        parts.append("ตัวเลือก:")
        for c in choices:
            c_clean = _clean(str(c))
            # strip leading garbage chars from OCR (e.g. \udc81. -> ก.)
            c_clean = re.sub(r"^[\x00-\x1f\ud800-\udfff]+\.?\s*", "", c_clean)
            if c_clean:
                parts.append(f"  {c_clean}")

    ans = _clean(obj.get("answer", ""))
    if ans:
        parts.append(f"คำตอบ: {ans}")

    sol = _clean(obj.get("solution", ""))
    if sol:
        parts.append(f"วิธีทำ: {sol}")

    return "\n".join(parts)

# ---- main conversion ----
dataset_dir  = Path(EXAMS_CLONE_DIR) / "dataset"
exams_docs   = []
stats        = {}
seen_texts   = set()   # simple exact-dedup within this source
exam_index   = 0

for fname in tqdm(EXAMS_JSONL_FILES, desc="JSONL files"):
    fpath = dataset_dir / fname
    if not fpath.exists():
        print(f"  SKIP (not found): {fname}")
        continue

    file_count  = 0
    file_skip   = 0

    with open(fpath, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = _build_text(obj)
            if len(text) < 20:
                file_skip += 1
                continue

            # exact-dedup: skip if same question already seen
            dedup_key = _clean(obj.get("question", ""))[:200]
            if dedup_key in seen_texts:
                file_skip += 1
                continue
            seen_texts.add(dedup_key)

            tr            = _thai_ratio(text)
            quality_flags = []
            if tr < 0.1:
                quality_flags.append("low_thai")

            source_name = obj.get("source_name", "")
            exam_type   = obj.get("exam_type", "")
            year        = str(obj.get("year", ""))
            grade       = obj.get("grade", "")

            doc = {
                "id":            f"{EXAMS_SOURCE}_{fname.replace('.jsonl','').replace('-','_')}_{exam_index:06d}",
                "source":        EXAMS_SOURCE,
                "license":       _clean(obj.get("license_note", "unknown")),
                "text":          text,
                "n_chars":       len(text),
                "n_words":       len(text.split()),
                "quality_flags": quality_flags,
                "dup_group":     None,
                "domain":        "math",
                "meta": {
                    "source_file":  fname,
                    "source_name":  source_name,
                    "exam_type":    exam_type,
                    "grade":        grade,
                    "year":         year,
                    "topic":        obj.get("topic", ""),
                    "has_image":    bool(obj.get("has_image", False)),
                    "thai_ratio":   round(tr, 4),
                },
            }
            exams_docs.append(doc)
            exam_index += 1
            file_count += 1

    stats[fname] = {"kept": file_count, "skipped": file_skip}

print(f"\nTotal docs : {len(exams_docs):,}")
print(f"Unique Q&A : {len(seen_texts):,}")
print()
print(f"{'File':<35} {'Kept':>6} {'Skipped':>8}")
print("-" * 55)
for fname, s in stats.items():
    print(f"{fname:<35} {s['kept']:>6,} {s['skipped']:>8,}")

In [ ]:
# Cell 16 - spot-check 5 random exam docs
import random

for i, doc in enumerate(random.sample(exams_docs, min(5, len(exams_docs)))):
    print(f"--- Doc {i+1} ---")
    print(f"id          : {doc['id']}")
    print(f"exam_type   : {doc['meta']['exam_type']}  grade: {doc['meta']['grade']}  year: {doc['meta']['year']}")
    print(f"flags       : {doc['quality_flags']}")
    print(f"thai_ratio  : {doc['meta']['thai_ratio']}")
    print(f"text:")
    print(doc["text"][:400])
    print()

In [ ]:
# Cell 17 - write exams docs to parquet
import math
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

out_path = Path(EXAMS_OUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

n_shards = max(1, math.ceil(len(exams_docs) / SHARD_SIZE))
written  = 0

for i in range(n_shards):
    chunk    = exams_docs[i * SHARD_SIZE : (i + 1) * SHARD_SIZE]
    shard_fn = out_path / f"thai_math_exams_{i:04d}.parquet"
    table    = pa.Table.from_pylist(chunk)
    pq.write_table(table, shard_fn, compression="snappy")
    written += len(chunk)
    print(f"  Shard {i:04d}: {len(chunk):,} docs -> {shard_fn.name}")

print(f"\nDone. {written:,} docs in {n_shards} shard(s) -> {out_path}")

In [ ]:
# Cell 18 - verify exams parquet + combined summary
import json
import pyarrow.parquet as pq
from datetime import datetime
from pathlib import Path

# verify
shards = sorted(Path(EXAMS_OUT_DIR).glob("*.parquet"))
if not shards:
    print("ERROR: no parquet files found")
else:
    table = pq.read_table(shards[0])
    print(f"Schema of {shards[0].name}:")
    print(table.schema)
    print(f"\nFirst doc:")
    print(json.dumps(table.to_pylist()[0], ensure_ascii=False, indent=2))

print()
print("=" * 60)
print("COMBINED SUMMARY - both repos")
print("=" * 60)

# mathayom corpus shards
c1_shards = sorted(Path(OUTPUT_DIR).glob("*.parquet"))
c1_rows   = sum(pq.read_metadata(s).num_rows for s in c1_shards)
c1_mb     = sum(s.stat().st_size for s in c1_shards) / 1e6

# exams shards
c2_shards = sorted(Path(EXAMS_OUT_DIR).glob("*.parquet"))
c2_rows   = sum(pq.read_metadata(s).num_rows for s in c2_shards)
c2_mb     = sum(s.stat().st_size for s in c2_shards) / 1e6

log = {
    "processed_at": datetime.now().isoformat(),
    "sources": [
        {
            "source":    "mathayom",
            "repo":      REPO_URL,
            "lang":      "Thai only",
            "n_docs":    c1_rows,
            "size_mb":   round(c1_mb, 2),
            "output":    str(Path(OUTPUT_DIR)),
            "granularity": "paragraph-level",
        },
        {
            "source":    "thai_math_exams",
            "repo":      EXAMS_REPO_URL,
            "lang":      "Thai",
            "n_docs":    c2_rows,
            "size_mb":   round(c2_mb, 2),
            "output":    str(Path(EXAMS_OUT_DIR)),
            "granularity": "Q&A record (question+choices+answer+solution)",
        },
    ],
    "total_docs": c1_rows + c2_rows,
    "total_mb":   round(c1_mb + c2_mb, 2),
    "schema":     "id,source,license,text,n_chars,n_words,quality_flags,dup_group,domain",
    "domain":     "math",
}

print(json.dumps(log, ensure_ascii=False, indent=2))